In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# unit and sanity checks!
"""--------------------------------------------"""
# add pooled r2 -> rerun movement r2 nb
# change epoch of movement avg

# replicate neurotheory plots
# > check the fits for different regularization constants
# define responsive

# one regressor
# add time
"""--------------------------------------------"""

## licks

In [ ]:
from sg.models import Encoder

fig, axes = plt.subplots(ncols=4, figsize=(5, 1.5), tight_layout=True)

encoder = Encoder(subj_id, sess_id)
encoder.get_r2()
encoder.plot_r2_comp(ax=axes[0])
axes[0].set_title("tv only")

encoder_response = Encoder(subj_id, sess_id, tv_keys=["response"])
encoder_response.get_r2()
encoder_response.plot_r2_comp(ax=axes[1])
axes[1].set_title("response only")

encoder_licks = Encoder(subj_id, sess_id, tv_keys=[], add_licks=True)
encoder_licks.get_r2()
encoder_licks.plot_r2_comp(ax=axes[2])
axes[2].set_title("licks only")

encoder_both = Encoder(subj_id, sess_id, add_licks=True)
encoder_both.get_r2()
encoder_both.plot_r2_comp(ax=axes[3])
axes[3].set_title("tv and licks")

In [ ]:
# wrap this up
# find the neuron where the residual of the task variable explained activity is most correlated with licks
from scipy.stats import pearsonr
import numpy as np

resids = encoder.robs - encoder.robs_predict["encoder"]
l_licks = encoder_licks.dm[:, encoder_licks.dm_idxs["n_left_licks"]]
r_licks = encoder_licks.dm[:, encoder_licks.dm_idxs["n_right_licks"]]
t_licks = l_licks + r_licks

licks = {"l": l_licks, "r": r_licks, "t": t_licks}

corr = {
    key: np.array(
        [pearsonr(resids[:, i], licks_).statistic for i in range(encoder.num_units)]
    )
    for key, licks_ in licks.items()
}

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    l_licks,
    resids[:, np.argmax(corr["l"])],
    xlabel="l licks",
    ylabel="resid spike count",
    add_lr=True,
)
plot_scatter(
    r_licks,
    resids[:, np.argmax(corr["r"])],
    xlabel="r licks",
    ylabel="resid spike count",
    add_lr=True,
)

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    x=encoder.scores["encoder"],
    y=encoder_response.scores["encoder"],
    xlabel=r"$r^2$, encoder (tv)",
    ylabel=r"$r^2$, encoder (response)",
    add_unity=True,
)
plot_scatter(
    x=encoder_response.scores["encoder"],
    y=encoder_licks.scores["encoder"],
    xlabel=r"$r^2$, encoder (response)",
    ylabel=r"$r^2$, encoder (licks)",
    add_unity=True,
)
plot_scatter(
    x=encoder.scores["encoder"],
    y=encoder_both.scores["encoder"],
    xlabel=r"$r^2$, encoder (tv)",
    ylabel=r"$r^2$, encoder (tv+licks)",
    add_unity=True,
)

In [ ]:
# sanity check that individual units with large slopes have correspondingly high weights
from scipy.stats import pearsonr

corr = {
    side: {
        reg: [
            pearsonr(
                encoder_licks.robs[:, i],
                encoder_licks.dm[:, encoder_licks.dm_idxs[f"n_{side}_licks"]],
            ).statistic
            for i in encoder_licks.reg_idxs[reg]
        ]
        for reg in encoder_licks.regions
    }
    for side in ["left", "right"]
}

In [ ]:
import numpy as np
from core.viz import plot_scatter


def plot(side="left", reg="DLS", i=0):
    i = np.argsort(corr[side][reg])[i]

    x = encoder_licks.dm[:, encoder_licks.dm_idxs[f"n_{side}_licks"]]
    y = encoder_licks.robs[:, encoder_licks.reg_idxs[reg][i]]

    fig, ax = plt.subplots(figsize=(2, 2))

    sc_dict = {"mean": {}, "std": {}}
    for n_licks in np.unique(x):
        scs = y[np.where(n_licks == x)[0]]
        sc_mean = np.mean(scs)
        sc_std = np.std(scs)

        sc_dict["mean"][n_licks] = sc_mean
        sc_dict["std"][n_licks] = sc_std

    ax.errorbar(
        x=np.unique(x),
        y=list(sc_dict["mean"].values()),
        yerr=list(sc_dict["std"].values()),
    )
    ax.set_title(f"{corr[side][reg][i]:.3f}")


plot(i=1)

In [ ]:
from core.viz import plot_scatter

# no systematic difference between adding licks and r
_, ax = plt.subplots(figsize=(2, 2))
plot_scatter(
    encoder.scores["encoder"], encoder_licks.scores["encoder"], add_unity=True, ax=ax
)

In [ ]:
encoder.verify()
encoder_licks.verify()

## init

In [ ]:
from sg.models import Encoder, StrategyEncoder

ns = [20, 50, 100, 200, 203]

encoders = {}

for n in ns:
    encoder = Encoder(subj_id, sess_id, n=n)
    encoder.get_r2()
    encoders[n] = encoder

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

In [ ]:
import numpy as np

scores = {n: encoder.scores["encoder"] for n, encoder in encoders.items()}

np.mean(list(scores.values()), axis=1)

In [ ]:
scores = {
    model: {n: encoder.scores[model] for n, encoder in encoders.items()}
    for model in ["baseline", "encoder"]
}

colors = {"baseline": "#666666", "encoder": "#DD9C11"}

fig, ax = plt.subplots(tight_layout=True)
for model, s_ in scores.items():
    ax.errorbar(
        x=s_.keys(),
        y=np.mean(list(s_.values()), axis=1),
        yerr=np.std(list(s_.values()), axis=1),
        color=colors[model],
        capsize=2,
        label=model,
    )

ax.axhline(y=0, color="#000000", linestyle="-", linewidth=0.5)

ax.set_xlabel("n. trials")
ax.set_ylabel(r"$r^2$")
ax.legend()

In [ ]:
encoder.verify()

In [ ]:
# check that nans are still caught as different between mb/mf with out of pool averging

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

In [ ]:
# se = ShuffledEncoder(
#     subj_id,
#     sess_id,
#     tv_keys=[
#         "response",
#         "rewarded",
#         "block_side",
#         "strategy",
#         "response_prev",
#         "rewarded_prev",
#     ],
# )
# se.plot_cvr2()
# se.plot_dr2()|
# se.plot_bound_r2()

## experiment

In [ ]:
encoder.reg_idxs["DLS"].shape, encoder.reg_idxs["DMS"].shape

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes

# scores
scores = {
    f"{reg}, {model}": encoder.scores[model][encoder.reg_idxs[reg]]
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}
# scores_mb = {f"{reg}, {model}": encoder_mb.scores[model][encoder_mb.reg_idxs[reg]] for reg in encoder_mb.regions for model in ['baseline', 'encoder']}
scores_mf = {
    f"{reg}, {model}": encoder_mf.scores[model][encoder_mf.reg_idxs[reg]]
    for reg in encoder_mf.regions
    for model in ["baseline", "encoder"]
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
colors = {"DMS": "#562E9C", "DLS": "#009D51"}
styles = {
    f"{reg}, {model}": {"linestyle": linestyles[model], "color": colors[reg]}
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

plot_kdes(scores, xlim=(-0.25, 1), line_kwargs=styles)
plot_kdes(scores_mf, xlim=(-0.25, 1), line_kwargs=styles)
# plot_kdes({f"{reg}, {model}": encoder_mb.scores[model][encoder_mb.reg_idxs[reg]] for reg in encoder_mb.regions for model in ['baseline', 'encoder']})